# Unit 4 — Supervised Fine-Tuning (SFT)

> *Part of the [RL for Robotics & LLMs](https://github.com/AliBuildsAI/rl-for-robotics-llms) series.*

Units 1–3 built policy gradients on robots, where the **environment gives a
reward**.  Units 4–8 move to language models, where it does not.  But before any
RL, there is a supervised step that everyone does first: **SFT**.

A *pretrained* language model knows grammar and facts, but it only does one
thing — predict the next token of internet text.  Ask it a question and it may
continue the question, or drift, rather than *answer*.  **Supervised
fine-tuning** teaches it to follow instructions by training on
(instruction, good-response) pairs.

This is the first hands-on stage of the RLHF pipeline, and the model we train
here is the **starting point for the rest of the series**.

> **Code provenance.**  The training code in Part B is **Nathan Lambert's
> reference SFT implementation**, ported *verbatim* from the RLHF book's
> [`code/instruction_tuning`](https://github.com/natolambert/rlhf-book/tree/main/code/instruction_tuning) (`config.py`, `utils.py`, `train.py`).
> Our only changes are presentational: drop his Weights & Biases + `rich`
> logging, reproduce his `Config` as a plain `dataclass` (same fields/defaults),
> add a matplotlib loss curve, and insert two clearly-marked *teaching asides*.
> The training recipe — model, data, masking, loss, optimiser, schedule,
> sampling — is entirely his, so it trains exactly as the book documents.

**What we build:**

| Part | What | Outcome |
|------|------|---------|
| A | Theory: the pipeline, chat templates, completion-only masking | The concepts |
| B | Nathan's SFT code, run end-to-end on OLMo-2-1B + No Robots | An instruction-following model |
| C | Watch it learn (in-loop samples) + the loss curve | Rambling → clean answers |


---
## 1  Where SFT Sits — The Pipeline

A modern chat model is built in three stages:

| Stage | Trains on | Result |
|-------|-----------|--------|
| **1. Pretraining** | Trillions of internet tokens (next-token prediction) | Knows language & facts, but doesn't follow instructions |
| **2. SFT** *(this unit)* | (instruction, response) demonstrations | Follows instructions, as well as the demos |
| **3. RLHF** *(Units 5–8)* | Preferences / rewards | Goes *beyond* the demonstrations |

SFT is the bridge from a raw pretrained model to something that behaves like an
assistant.  It is also the **ceiling-setter for imitation**: the model can only
be as good as the responses it's shown.  That ceiling is why RLHF exists — to
push past "imitate the demo" toward "produce what people actually prefer."  But
you need a competent instruction-follower first, and that's SFT.


---
## 2  What SFT Actually Is — Next-Token Prediction, Curated

Here is the part that surprises people: **SFT uses the exact same loss as
pretraining** — cross-entropy on the next token. Nothing about the objective is
new.  Two things change:

1. **The data.** Instead of random internet text, we train on curated
   (instruction → response) pairs, formatted with the model's **chat template**
   (the special tokens that mark *user* and *assistant* turns).

2. **What we compute the loss on.** We only want the model to learn to produce
   the **response**, not to re-generate the user's prompt.  So we **mask the
   prompt tokens out of the loss** and train only on the assistant's tokens.

That second point is the one genuinely SFT-specific idea, and it's a favourite
interview question. The next section makes it concrete.


### Chat templates — the format SFT trains on

How does a model know where the user stops and the assistant starts?  A **chat
template** wraps each turn in special tokens.  A common one is **ChatML** (three
roles — *system*, *user*, *assistant*), which Qwen uses:

```
<|im_start|>user
How many helicopters can a human eat in one sitting?<|im_end|>
<|im_start|>assistant
```

**OLMo-2 (our model) uses its own markers** (`<|user|>`, `<|assistant|>`, and it
reuses `<|endoftext|>` as BOS/EOS) — we print its actual template below.  Two
points hold for any template:

- The trailing **assistant header** is the **generation prompt** — where the
  model begins writing (`apply_chat_template(..., add_generation_prompt=True)`).
- Packing all data into one consistent template makes the **end-of-turn token** a
  clean, learnable "stop" signal.


---
## 3  The Key Detail — Completion-Only Loss Masking

We feed the whole formatted conversation to the model, but for the **labels** we
replace every token belonging to the prompt (system/user, up to and including the
assistant header) with the ignore-index **`-100`**.  PyTorch's cross-entropy
skips any position whose label is `-100`, so **no gradient flows from the prompt
tokens** — the model is graded only on the response it should have produced.

Why it matters:

- Without masking, the model would also be trained to generate user prompts —
  wasting capacity and teaching the wrong behaviour.
- The masked positions still serve as **context** (the model attends to them);
  they just don't contribute to the loss.

This is exactly what Nathan's `_encode_row` does — we'll see the `-100`s on a
real example once his code is loaded.


---
## 4  Setup

A modern (Ampere-or-newer) GPU is assumed — A100, H100, L4, or an RTX 30/40/50
card (incl. the **4090**), all of which support **bf16**.  We train **OLMo-2-1B**
(batch 4, 2048-length, gradient checkpointing), which needs roughly **~14–18 GB**
in bf16 — comfortable on any 24 GB+ card and trivial on an A100/H100.


In [ ]:
%pip install -q "transformers>=4.45" "datasets>=3.0" accelerate matplotlib
print("packages ready")


### Imports

These are the imports from Nathan's `utils.py` / `train.py` (minus `wandb` and
`rich`, which we don't use here), plus `matplotlib` for the loss curve.


In [ ]:
import os
import platform
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils import clip_grad_norm_
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizer
import matplotlib.pyplot as plt

print(f"torch {torch.__version__}  |  cuda: {torch.cuda.is_available()}")


---
## Part B — Nathan Lambert's SFT Code, Ported Verbatim

Everything below is his reference implementation from
[`code/instruction_tuning`](https://github.com/natolambert/rlhf-book/tree/main/code/instruction_tuning).  Each cell says which of his files /
functions it comes from.  The only edits are cosmetic (no W&B, no `rich`).


### `Config` — from `config.py`

His full training configuration (originally a `pydantic` model; reproduced here
as a plain `dataclass` with the **same fields and defaults**).  Key values:
OLMo-2-1B base, No Robots, LR `5e-6`, 3 epochs, effective batch `4×8 = 32`,
`max_length 2048`, gradient checkpointing, and in-loop sampling every 50 steps.

> **One knob to know:** `bf16=True` loads the model in bfloat16 (his default). If
> your run shows the loss stuck flat (~2.3) instead of dropping, your optimiser is
> underflowing bf16 weight updates — set `cfg.bf16 = False` for fp32 and re-run.
> On modern GPUs his bf16 default is the intended setting.


In [ ]:
@dataclass
class Config:
    # Model
    model_name: str = "allenai/OLMo-2-0425-1B"
    chat_template_source: str | None = "allenai/OLMo-2-0425-1B-SFT"
    # Data
    dataset_name: str = "HuggingFaceH4/no_robots"
    dataset_split: str = "train"
    max_samples: int | None = None
    max_length: int = 2048
    # Training
    lr: float = 5.0e-6
    num_epochs: int = 3
    batch_size: int = 4
    gradient_accumulation_steps: int = 8
    warmup_ratio: float = 0.1
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    seed: int = 42
    # Hardware
    bf16: bool = True
    gradient_checkpointing: bool = True
    model_device_id: int = 0
    # In-loop generation
    sample_every: int = 50
    sample_max_tokens: int = 128
    sample_max_input_tokens: int = 512
    sample_temperature: float = 0.7
    sample_top_p: float = 0.9
    sample_do_sample: bool = True

cfg = Config()

if torch.cuda.is_available():
    device = torch.device(f"cuda:{cfg.model_device_id}")
else:
    device = torch.device("cpu")
print("device:", device)


### Helpers — from `utils.py`

`seed_everything` (full reproducibility), `get_attn_implementation` (uses
Flash-Attention-2 if installed, else PyTorch's `sdpa`), the ignore index, and the
fixed prompts sampled during training.


In [ ]:
IGNORE_INDEX = -100

DEFAULT_SAMPLE_PROMPTS: list[str] = [
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
    "Write a haiku about programming.",
    "How does photosynthesis work?",
]

def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_attn_implementation() -> str:
    if platform.machine() != "x86_64":
        return "sdpa"
    try:
        import flash_attn  # noqa: F401
        return "flash_attention_2"
    except ImportError:
        return "sdpa"


### `load_model` — from `utils.py`

**In:** the config + device.  **Out:** `(model, tokenizer)`.

What it does, line by line:
- loads the **base** OLMo-2-1B tokenizer and, because a base tokenizer has **no
  chat template**, lifts one from the SFT sibling (`chat_template_source`);
- sets a pad token if missing;
- loads the model in **bf16** (his default) with the chosen attention kernel;
- enables **gradient checkpointing** (`use_reentrant=False`) to fit 2048-length
  sequences in less VRAM.


In [ ]:
def load_model(cfg: Config, device: torch.device):
    attn_impl = get_attn_implementation()
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=False)

    if tokenizer.chat_template is None and cfg.chat_template_source:
        donor = AutoTokenizer.from_pretrained(cfg.chat_template_source, trust_remote_code=False)
        if donor.chat_template is None:
            raise ValueError(
                f"chat_template_source {cfg.chat_template_source} has no chat_template."
            )
        tokenizer.chat_template = donor.chat_template

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if cfg.bf16 else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        trust_remote_code=False,
        attn_implementation=attn_impl,
        torch_dtype=dtype,
    ).to(device)

    if cfg.gradient_checkpointing:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

    return model, tokenizer


model, tokenizer = load_model(cfg, device)
print(f"loaded {cfg.model_name} | {sum(p.numel() for p in model.parameters())/1e9:.2f}B params")


> **Aside (our teaching add-on, not in Nathan's script).**  Let's see OLMo's
> actual chat template and its special tokens.  Every chat model wraps turns in
> **special role-marker tokens** added to the vocabulary; the role names and the
> message text are ordinary tokens.


In [ ]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": "How many helicopters can a human eat in one sitting?"}],
    tokenize=False, add_generation_prompt=True)
print("=== formatted prompt (note OLMo's special role markers) ===")
print(text)
print("\nspecial tokens:", tokenizer.all_special_tokens)


### The data pipeline — from `utils.py`

Four pieces, all his:

- **`_encode_row`** — the masking.  Tokenise the prompt (all messages except the
  final assistant turn, `add_generation_prompt=True`) and the full conversation,
  then set `labels = [-100]*len(prompt) + response_ids`.  Skips rows whose last
  turn isn't `assistant` or that end up fully masked.  Returns **tensors**.
  *(Note the `return_dict=False` on `apply_chat_template` — it forces a plain
  `list[int]`; without it some versions return an `Encoding` object.)*
- **`SFTDataset`** — a tiny `torch.utils.data.Dataset` wrapping the encoded list.
- **`_collate`** — pads a batch: `input_ids` with the pad id, `attention_mask`
  with 0, and **`labels` with `-100`** (padding is never trained on either).
- **`create_dataloader`** — loads No Robots, encodes every row **in Python** (no
  `datasets.map`, so nothing is serialised to Arrow), and returns a `DataLoader`.

Note `SFTBatch`/`compute_loss` too — the batch dataclass and the loss.


In [ ]:
@dataclass
class SFTBatch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    labels: torch.Tensor

    def to(self, device: torch.device | str) -> "SFTBatch":
        return SFTBatch(
            input_ids=self.input_ids.to(device),
            attention_mask=self.attention_mask.to(device),
            labels=self.labels.to(device),
        )


def compute_loss(model, batch: "SFTBatch") -> torch.Tensor:
    """Causal-LM cross-entropy with prompt-masked labels (-100 ignored)."""
    out = model(input_ids=batch.input_ids, attention_mask=batch.attention_mask, use_cache=False)
    shift_logits = out.logits[:, :-1, :].contiguous()
    shift_labels = batch.labels[:, 1:].contiguous()
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=IGNORE_INDEX,
    )


def _encode_row(messages: list[dict], tokenizer: PreTrainedTokenizer, max_length: int):
    """Render `messages` with the chat template and mask all but the final assistant turn."""
    if not messages or messages[-1]["role"] != "assistant":
        return None

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True, return_dict=False
    )
    full_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False, return_dict=False
    )
    labels = [IGNORE_INDEX] * len(prompt_ids) + list(full_ids[len(prompt_ids):])

    if max_length is not None and len(full_ids) > max_length:
        full_ids = full_ids[:max_length]
        labels = labels[:max_length]

    if all(label == IGNORE_INDEX for label in labels):
        return None

    return {
        "input_ids": torch.tensor(full_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


class SFTDataset(Dataset):
    def __init__(self, encoded):
        self.encoded = encoded

    def __len__(self) -> int:
        return len(self.encoded)

    def __getitem__(self, idx: int):
        return self.encoded[idx]


def _collate(examples, pad_token_id: int) -> SFTBatch:
    max_len = max(ex["input_ids"].size(0) for ex in examples)
    input_ids, attention_mask, labels = [], [], []
    for ex in examples:
        ids, lbl = ex["input_ids"], ex["labels"]
        pad = max_len - ids.size(0)
        input_ids.append(torch.cat([ids, torch.full((pad,), pad_token_id, dtype=torch.long)]))
        attention_mask.append(
            torch.cat([torch.ones(ids.size(0), dtype=torch.long), torch.zeros(pad, dtype=torch.long)])
        )
        labels.append(torch.cat([lbl, torch.full((pad,), IGNORE_INDEX, dtype=torch.long)]))
    return SFTBatch(
        input_ids=torch.stack(input_ids),
        attention_mask=torch.stack(attention_mask),
        labels=torch.stack(labels),
    )


def create_dataloader(cfg: Config, tokenizer: PreTrainedTokenizer) -> DataLoader:
    raw = load_dataset(cfg.dataset_name, split=cfg.dataset_split)
    if cfg.max_samples is not None and len(raw) > cfg.max_samples:
        raw = raw.select(range(cfg.max_samples))

    encoded = []
    skipped = 0
    for example in raw:
        row = _encode_row(example["messages"], tokenizer, cfg.max_length)
        if row is None:
            skipped += 1
            continue
        encoded.append(row)

    if not encoded:
        raise RuntimeError("No trainable rows after tokenization.")
    if skipped:
        print(f"Skipped {skipped}/{len(raw)} rows (no trainable assistant tokens).")

    pad_id = tokenizer.pad_token_id
    return DataLoader(
        SFTDataset(encoded),
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=lambda batch: _collate(batch, pad_id),
        num_workers=0,
        pin_memory=False,
    )


> **Aside — see the `-100`s.**  Run `_encode_row` on one real No-Robots example
> and print the first tokens with their labels: the prompt tokens are `-100`
> (masked), the response tokens carry real ids (trained).


In [ ]:
_probe = load_dataset(cfg.dataset_name, split=cfg.dataset_split)[0]["messages"]
_row = _encode_row(_probe, tokenizer, cfg.max_length)
_ids, _labels = _row["input_ids"].tolist(), _row["labels"].tolist()
_masked = sum(l == IGNORE_INDEX for l in _labels)
print(f"{len(_ids)} tokens  |  {_masked} masked (prompt)  |  {len(_ids)-_masked} trained (response)\n")
print(f"{'token':>18} | label")
print("-" * 34)
for tok_id, lab in list(zip(_ids, _labels))[:32]:
    tok = tokenizer.decode([tok_id]).replace("\n", "\\n")
    print(f"{tok!r:>18} | {'-100 (masked)' if lab == IGNORE_INDEX else 'train'}")


In [ ]:
dataloader = create_dataloader(cfg, tokenizer)
print(f"{len(dataloader.dataset)} training examples  |  {len(dataloader)} batches/epoch")


### Scheduler + sampling — from `utils.py`

- **`make_lr_scheduler`** — linear warmup (10% of steps) then linear decay to 0.
- **`generate_samples`** — the key monitoring trick: every `sample_every` steps,
  generate from a few fixed prompts and print them.  For SFT this is *the* signal
  (not val loss) — you literally watch the model go from rambling to a single
  clean, terminated answer.  (His version pretty-prints with `rich`; we plain-print
  and keep `skip_special_tokens=False` so you can see the role markers.)


In [ ]:
def make_lr_scheduler(optimizer, total_steps: int, warmup_ratio: float):
    warmup_steps = int(total_steps * warmup_ratio)

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps + 1))
        remaining = total_steps - step
        return max(0.0, remaining / max(1, total_steps - warmup_steps))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def generate_samples(model, tokenizer, cfg, step, prompts=None, max_new_tokens=None):
    was_training = model.training
    model.eval()
    new_tokens = max_new_tokens if max_new_tokens is not None else cfg.sample_max_tokens
    if prompts is None:
        prompts = DEFAULT_SAMPLE_PROMPTS

    print(f"\n{'='*72}\nSamples @ step {step}\n{'='*72}")
    for i, prompt in enumerate(prompts, start=1):
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(
            formatted, return_tensors="pt", truncation=True, max_length=cfg.sample_max_input_tokens
        ).to(model.device)
        kwargs = dict(
            **inputs,
            max_new_tokens=new_tokens,
            do_sample=cfg.sample_do_sample,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
        if cfg.sample_do_sample:
            kwargs.update(temperature=cfg.sample_temperature, top_p=cfg.sample_top_p)
        with torch.no_grad():
            out = model.generate(**kwargs)
        full_text = tokenizer.decode(out[0], skip_special_tokens=False)
        print(f"\n[Prompt {i}] {prompt}\n{full_text}")

    if was_training:
        model.train()


### The training loop — from `train.py`

His `main()` loop, verbatim except W&B → a `loss_log` list + prints:

- **AdamW**, the linear-warmup/decay schedule, gradient accumulation to effective
  batch 32;
- a per-batch **`loss.isfinite()`** guard (skips a bad batch instead of crashing);
- at each accumulation boundary: **sample generations** (at step 0 you see the
  *base* model, then it improves), clip gradients, `optimizer.step()`,
  `scheduler.step()`;
- runs `sample_every=50`, so expect samples at steps 0, 50, 100, ….

**What to watch:** the printed **train loss should drop** (sharp early, then
gradual — the book's curve), and the **step-0 → later samples** should visibly go
from rambling to a clean, terminated answer.  (SFT is judged by *generations*, not
held-out loss — which is why Nathan samples instead of computing eval perplexity.)


In [ ]:
seed_everything(cfg.seed)

accum = cfg.gradient_accumulation_steps
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
steps_per_epoch = len(dataloader) // accum
total_steps = steps_per_epoch * cfg.num_epochs
warmup_steps = int(total_steps * cfg.warmup_ratio)
scheduler = make_lr_scheduler(optimizer, total_steps, cfg.warmup_ratio)

print(f"Effective batch {cfg.batch_size}x{accum}={cfg.batch_size*accum}  |  "
      f"{total_steps} steps ({warmup_steps} warmup)")

loss_log = []                       # (our addition — for the loss curve, replaces W&B)
global_step = 0
model.train()
optimizer.zero_grad(set_to_none=True)
accumulated_loss = 0.0
micro_in_step = 0

for epoch in range(cfg.num_epochs):
    print(f"\n===== Epoch {epoch+1}/{cfg.num_epochs} =====")
    for batch_idx, batch in enumerate(dataloader):
        batch = batch.to(device)
        loss = compute_loss(model, batch)
        if loss.isfinite():
            scaled = loss / accum
            scaled.backward()
            accumulated_loss += loss.item()
            micro_in_step += 1

        if (batch_idx + 1) % accum == 0:
            if cfg.sample_every > 0 and global_step % cfg.sample_every == 0:
                generate_samples(model, tokenizer, cfg, step=global_step)

            grad_norm = clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            avg_loss = accumulated_loss / max(1, micro_in_step)
            loss_log.append((global_step, avg_loss, float(grad_norm)))
            if global_step % 25 == 0:
                print(f"step {global_step}/{total_steps}  loss {avg_loss:.4f}  "
                      f"grad_norm {float(grad_norm):.2f}  lr {scheduler.get_last_lr()[0]:.2e}")
            accumulated_loss = 0.0
            micro_in_step = 0

    optimizer.zero_grad(set_to_none=True)
    accumulated_loss = 0.0
    micro_in_step = 0

print("\nSFT complete.")


---
## Part C — The Training Curves

Two panels:

- **Training loss** — the light line is the per-step loss (noisy); the bold line
  is a 25-step moving average.  Expect a **sharp early drop** (~first 100–150
  steps, learning the chat template) then a **plateau**.  It's noisy because each
  point is the loss on a *different random batch* of 32 examples — **read the
  trend, not the individual points.**
- **Gradient norm** — how big each update was *before* clipping (clipped at
  `max_grad_norm=1.0`).  Largest early (fast learning), settling as it converges.

For SFT the loss floor is **not** the success metric — the **generations** above
are.  The loss just confirms the model learned the format.


In [ ]:
steps  = [x[0] for x in loss_log]
losses = [x[1] for x in loss_log]
gnorms = [x[2] for x in loss_log]

def smooth(y, w=25):
    out = []
    for i in range(len(y)):
        lo = max(0, i - w + 1)
        out.append(sum(y[lo:i + 1]) / (i - lo + 1))
    return out

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax1.plot(steps, losses, alpha=0.3, label="per-step loss (noisy)")
ax1.plot(steps, smooth(losses), linewidth=2, label="25-step moving average")
ax1.set_ylabel("training loss")
ax1.set_title("SFT training loss — read the trend, not the noise")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(steps, gnorms, alpha=0.5, color="darkorange")
ax2.axhline(cfg.max_grad_norm, ls="--", color="gray",
            label=f"clip threshold ({cfg.max_grad_norm})")
ax2.set_xlabel("optimiser step"); ax2.set_ylabel("grad norm (pre-clip)")
ax2.set_title("Gradient norm — update size before clipping")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.savefig("sft_curves.png", dpi=130); plt.show()

print(f"train loss (smoothed): {smooth(losses)[0]:.3f} -> {smooth(losses)[-1]:.3f}")
print(f"grad norm  (smoothed): {smooth(gnorms)[0]:.2f} -> {smooth(gnorms)[-1]:.2f}")


### Save the checkpoint

This SFT model is the artifact the rest of the series builds on.

- `model.save_pretrained(dir)` / `tokenizer.save_pretrained(dir)` — **write** the
  model + tokenizer to a self-contained folder you can later `from_pretrained`.


In [ ]:
SAVE_DIR = "olmo-sft-final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved SFT model + tokenizer -> {SAVE_DIR}/")


---
## 5  What's Next — Reward Models (Unit 5)

We now have an instruction-following model — but it only knows how to *imitate*
the responses it was shown.  It has no concept of one answer being **better** than
another.

That's the gap **Unit 5** fills: we collect human **preference** comparisons and
train a **reward model** $r_\phi(x, y)$ that scores any response.  Then Units 6–8
use RL — PPO, then DPO and GRPO — to push past imitation toward responses people
actually prefer.

→ [Back to the series](https://github.com/AliBuildsAI/rl-for-robotics-llms)
